# Machine Learning Project for California Housing 
Based on _Hands-on Machine Learning_ by Aurelien Geron


In [ ]:
from sklearn.model_selection import train_test_split
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


## Load Data

In [ ]:
housing_full = pd.read_csv(Path("data/housing.csv"))

## Visualise Data

In [ ]:
plt.rc("font", size=8)
housing_full.hist(bins=50, figsize=(12, 8))

Observations: 
- median age and median income are capped. This may be wrongly learnt by the model
- many features are right-skewed, which is not good for machine learning
- total_bedrooms have null values 
- different features have very different scales

## Train Test Split
- you can pass it multiple datasets with an identical number of rows, and it will split them on the same indices
- splitting randomly here is fine, but it may not be when population size is small
- here we are creating a new feature `income_cat` for use in stratified sampling

In [ ]:
housing_full["income_cat"] = pd.cut(housing_full["median_income"], 
                                    bins=[0., 1.5, 3.0, 4.5, 6.0, np.inf],
                                    labels=[1, 2, 3, 4, 5])


Visualisation of income_cat

In [ ]:
housing_full["income_cat"].value_counts().sort_index().plot(kind="bar")
plt.show()

Comparing stratified sampling and random sampling
- apparently the composition of test_1 is closer to housing_full

In [ ]:
# splitting randomly 
train, test = train_test_split(housing_full, test_size=0.2, random_state=42)

# splitting with stratified sampling 
train_1, test_1 = train_test_split(housing_full, test_size=0.2, random_state=42, stratify=housing_full["income_cat"])

# print(test["income_cat"].value_counts().sort_index() / len(test))
# print(test_1["income_cat"].value_counts().sort_index() / len(test))
# print(housing_full["income_cat"].value_counts().sort_index() / len(housing_full))

# now income_cat is useless, we drop it 
for set in (train_1, test_1): 
    set.drop("income_cat", axis=1, inplace=True)

# make a copy of train_1 for future use 
housing: pd.DataFrame = train_1.copy()

## Data Visualisation
Visualise geographical data

In [ ]:
housing.plot(kind="scatter", x="longitude", y="latitude", grid=True, alpha=0.2)
plt.show()


Check correlations
- median house value is not correlated to population. Why? 

In [ ]:
corr_matrix = housing.corr(numeric_only=True)
corr_matrix["median_house_value"].sort_values(ascending=False)


Create scatter matrix

In [ ]:
from pandas.plotting import scatter_matrix
useful_attributes = ["median_house_value", "median_income", "total_rooms", "housing_median_age"]
scatter_matrix(housing[useful_attributes])
plt.show()

Plot a scatter of median house value and median income

In [ ]:
housing.plot(kind="scatter", x="median_income", y="median_house_value", alpha=0.2)

## Combine Attributes
- more rooms per house > higher value
- a larger percentage of rooms are bedrooms > lower value

In [ ]:
housing["rooms_per_house"] = housing["total_rooms"] / housing["households"]
housing["bedrooms_ratio"] = housing["total_bedrooms"] / housing["total_rooms"]
housing["people_per_house"] = housing["population"] / housing["households"]

useful_attributes_1 = ["rooms_per_house", "bedrooms_ratio", "people_per_house", "median_house_value"]
new_corr_matrix = housing[useful_attributes_1].corr()
print(new_corr_matrix["median_house_value"].sort_values(ascending=False))

## Prepare the data
- write functions instead of doing it manually
- so that the transformation is easily reproducible

In [ ]:
# start with a clean set of training data
housing = train_1.drop("median_house_value", axis=1)
housing_y = train_1["median_house_value"].copy()

Fill in missing total_bedrooms data
- Can use .fillna(), but better to use SimpleImputer
- It stores median value, which can be used on not only the training set

In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

# median can only be computed for numerical values
# so we drop ocean_proximity
housing_num = housing.select_dtypes(include=[np.number])
imputer.fit(housing_num)
X = imputer.transform(housing_num)

# X is a numpy array, not a pandas dataframe 
housing_tr = pd.DataFrame(X, columns=housing_num.columns, 
                          index=housing_num.index)
housing_tr.info()


Convert categorical attributes into numbers

In [ ]:
ocean_prox = housing[["ocean_proximity"]].copy()

from sklearn.preprocessing import OrdinalEncoder
ordinal_encoder = OrdinalEncoder()
ocean_prox_ordinal = ordinal_encoder.fit_transform(ocean_prox)

# the problem with ordinal encoder is, the model will assume that 
# closer values are more similar, but that is not the case here
# so we use one hot encoder
from sklearn.preprocessing import OneHotEncoder
onehot_encoder = OneHotEncoder()
ocean_prox_onehot = onehot_encoder.fit_transform(ocean_prox)
print(ocean_prox_onehot[:5]) # it is a scipy csr_matrix

Feature Scaling

In [ ]:
# 